In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, kruskal

sns.set_style("whitegrid")

# Seed para reprodutibilidade dos testes de permutação
RNG = np.random.default_rng(42)


In [ ]:
from genres_mask import GENRE_BY_NAME, Genre


In [ ]:
df = pd.read_parquet('db/movies.parquet')


In [ ]:
def bitmask_to_list(mask: int) -> list[str]:
    l = []
    for g in Genre:
        if mask & g:
            l.append(g.name)
    return l


In [ ]:
df['genres'] = df['genres'].apply(bitmask_to_list)


In [ ]:
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df2 = pd.read_parquet('db/ratings.parquet')

df_final = df.merge(df2)


In [ ]:
df_final.head()


In [ ]:
# Valores nulos por coluna
df_final.isnull().sum()


In [ ]:
# Remove listas vazias
df_genres = df_final[df_final["genres"].map(len) > 0].copy()

# Cria uma linha para cada gênero
df_genres = df_genres.explode("genres")

df_genres.head()


In [ ]:
# Gêneros com mais filmes

genre_count = (
    df_genres["genres"]
    .value_counts()
    .sort_values(ascending=False)
)

plt.figure(figsize=(9,6))

genre_count.plot(kind="bar")

plt.title("Quantidade de filmes por gênero")
plt.xlabel("Gênero")
plt.ylabel("Quantidade")

plt.xticks(rotation=45)

plt.show()


In [ ]:
# Média de avaliação por gênero

genre_rating = (
    df_genres
    .groupby("genres")["averageRating"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(9,6))

genre_rating.plot(kind="bar")

plt.title("Média de avaliações por gênero")
plt.xlabel("Gênero")
plt.ylabel("Average Rating")

plt.xticks(rotation=45)

plt.show()


In [ ]:
# Gêneros com mais engajamento

genre_votes = (
    df_genres
    .groupby("genres")["numVotes"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(9,6))

genre_votes.plot(kind="bar")

plt.title("Média de votos por gênero")
plt.xlabel("Gênero")
plt.ylabel("Número médio de votos")

plt.xticks(rotation=45)

plt.show()


In [ ]:
# Score de desempenho

genre_performance = (
    df_genres
    .groupby("genres")
    .agg({
        "averageRating": "mean",
        "numVotes": "mean"
    })
)

# Normalização simples
genre_performance["rating_norm"] = (
    genre_performance["averageRating"] /
    genre_performance["averageRating"].max()
)

genre_performance["votes_norm"] = (
    genre_performance["numVotes"] /
    genre_performance["numVotes"].max()
)

# Score final
genre_performance["performance_score"] = (
    genre_performance["rating_norm"] * 0.6 +
    genre_performance["votes_norm"] * 0.4
)

genre_performance = genre_performance.sort_values(
    by="performance_score",
    ascending=False
)

genre_performance


In [ ]:
plt.figure(figsize=(9,6))

plt.bar(
    genre_performance.index,
    genre_performance["performance_score"]
)

plt.title("Desempenho dos gêneros cinematográficos")
plt.xlabel("Gênero")
plt.ylabel("Score de desempenho")

plt.xticks(rotation=45)

plt.show()


In [ ]:
top_genres = genre_count.head(5).index

top_genres


In [ ]:
# BOOTSTRAP

bootstrap_results = {}

n_bootstrap = 1000

for genre in top_genres:

    ratings = df_genres[
        df_genres["genres"] == genre
    ]["averageRating"].dropna()

    means = []

    for _ in range(n_bootstrap):

        sample = ratings.sample(
            frac=1,
            replace=True
        )

        means.append(sample.mean())

    bootstrap_results[genre] = means


In [ ]:
plt.figure(figsize=(9,6))

for genre, values in bootstrap_results.items():

    sns.kdeplot(values, label=genre)

plt.title("Distribuição bootstrap da média das avaliações")
plt.xlabel("Average Rating")
plt.ylabel("Densidade")

plt.legend()

plt.show()


In [ ]:
# INTERVALO DE CONFIANÇA

confidence_intervals = []

for genre, values in bootstrap_results.items():

    lower = np.percentile(values, 2.5)
    upper = np.percentile(values, 97.5)

    confidence_intervals.append({
        "genre": genre,
        "lower": lower,
        "upper": upper,
        "mean": np.mean(values)
    })

ci_df = pd.DataFrame(confidence_intervals)

ci_df


In [ ]:
plt.figure(figsize=(9,6))

plt.errorbar(
    ci_df["genre"],
    ci_df["mean"],
    yerr=[
        ci_df["mean"] - ci_df["lower"],
        ci_df["upper"] - ci_df["mean"]
    ],
    fmt='o'
)

plt.title("Intervalo de confiança das avaliações por gênero")
plt.xlabel("Gênero")
plt.ylabel("Average Rating")

plt.show()


In [ ]:
plot_df = df_genres[
    df_genres["genres"].isin(top_genres)
]

plt.figure(figsize=(12,6))

sns.boxplot(
    data=plot_df,
    x="genres",
    y="averageRating"
)

plt.title("Distribuição das avaliações por gênero")
plt.xlabel("Gênero")
plt.ylabel("Average Rating")

plt.show()


# Teste de hipótese: gêneros têm avaliações diferentes?

**Pergunta**: Quais gêneros apresentam melhor desempenho combinado de popularidade (votos) e avaliação (nota)?

Primeiro testamos se as notas médias dos 5 gêneros mais populares diferem entre si de forma geral (**Kruskal-Wallis**, equivalente não-paramétrico da ANOVA para mais de 2 grupos).

H0: as distribuições de `averageRating` são iguais entre os 5 gêneros mais populares.

Se houver diferença significativa, comparamos especificamente o gênero com **melhor** e o com **pior** `performance_score` via Mann-Whitney e teste de permutação.


In [ ]:
# KRUSKAL-WALLIS entre os top 5 gêneros

groups = [
    plot_df[plot_df["genres"] == genre]["averageRating"].dropna()
    for genre in top_genres
]

h_stat, p_kruskal = kruskal(*groups)

print("="*60)
print("KRUSKAL-WALLIS - AVALIAÇÃO ENTRE OS 5 GÊNEROS MAIS POPULARES")
print("="*60)
print(f"\nGêneros comparados: {list(top_genres)}")
print(f"H = {h_stat:.3f}")
print(f"p-valor = {p_kruskal:.5f}")

if p_kruskal < 0.05:
    print("Conclusão: Há diferença estatisticamente significativa "
          "na avaliação entre pelo menos um dos gêneros.")
else:
    print("Conclusão: Não há diferença estatisticamente significativa "
          "na avaliação entre os gêneros.")


In [ ]:
# COMPARAÇÃO PAR A PAR: melhor x pior performance_score

best_genre = genre_performance.index[0]
worst_genre = genre_performance.index[-1]

print(f"Melhor performance_score: {best_genre}")
print(f"Pior performance_score : {worst_genre}")

ratings_best = df_genres[
    df_genres["genres"] == best_genre
]["averageRating"].dropna()

ratings_worst = df_genres[
    df_genres["genres"] == worst_genre
]["averageRating"].dropna()


In [ ]:
def permutation_test_diff_means(a, b, n_perm=10_000, rng=RNG):
    """
    Teste de permutação para diferença de médias entre dois grupos.
    H0: não há diferença entre os grupos (rótulo é irrelevante).
    Retorna a diferença observada e o p-valor bicaudal.
    """
    a = np.asarray(a)
    b = np.asarray(b)

    observed_diff = a.mean() - b.mean()

    pooled = np.concatenate([a, b])
    n_a = len(a)

    perm_diffs = np.empty(n_perm)

    for i in range(n_perm):
        shuffled = rng.permutation(pooled)
        perm_diffs[i] = shuffled[:n_a].mean() - shuffled[n_a:].mean()

    p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))

    return observed_diff, p_value


In [ ]:
u_stat, p_mw = mannwhitneyu(
    ratings_best,
    ratings_worst,
    alternative="two-sided"
)

diff_obs, p_perm = permutation_test_diff_means(ratings_best, ratings_worst)

print("="*60)
print(f"{best_genre} x {worst_genre}")
print("="*60)
print(f"\nMédia {best_genre} : {ratings_best.mean():.3f}")
print(f"Média {worst_genre} : {ratings_worst.mean():.3f}")
print(f"Diferença observada: {diff_obs:.3f}")
print(f"\nMann-Whitney        : p = {p_mw:.5f}")
print(f"Permutação (10.000) : p = {p_perm:.5f}")

if p_mw < 0.05:
    print("\nConclusão: A diferença entre os dois gêneros é "
          "estatisticamente significativa.")
else:
    print("\nConclusão: A diferença entre os dois gêneros não é "
          "estatisticamente significativa.")
